# ema-second-moment composite — cx26: v buffer updated via in-place copy_ (no rebinding)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `buffer-copy_-inplace`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-second-moment"
DD_ATOM_IDS = ["ema-second-moment", "buffer-copy_-inplace"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "PyTorch: in-place buffer copy"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's second-moment `v` is a per-parameter buffer that *evolves* across steps. The naive way to update it is `v = beta2 * v + (1 - beta2) * g * g`, but this REBINDS the local name to a new tensor — the original buffer (registered via `register_buffer` or held in an optimizer state dict) keeps its old value forever.

The fix: compute the new value out-of-place, then **copy_** it back into the same storage.

**The two atoms.**
- **ema-second-moment** — the formula `v_new = beta2 * v + (1 - beta2) * g**2`.
- **buffer-copy_-inplace** — `v.copy_(v_new)` rather than `v = v_new`. `copy_` writes into the EXISTING tensor's storage, so any reference held elsewhere (`state_dict`, `register_buffer` entry, optimizer dict) sees the update.

**Anatomy.**
```python
v_new = beta2 * v + (1 - beta2) * g * g     # ema-second-moment (out-of-place compute).
v.copy_(v_new)                              # buffer-copy_-inplace (write back).
```

This composition is the *manual* pattern; `v.mul_(beta2).addcmul_(g, g, value=1-beta2)` is the more efficient fused-in-place pattern. We test the EXPLICIT compute-then-copy pattern here because that's what makes the buffer-aliasing contract visible.

### Composite Exercise — v buffer updated via in-place copy_ (no rebinding)

**Atoms exercised together**: `ema-second-moment`, `buffer-copy_-inplace`

Implement `cx26_update_v_via_copy(v, g, beta2)`.

Required behaviour:
1. Compute `v_new = beta2 * v + (1 - beta2) * g * g` (the second-moment EMA, atom: ema-second-moment).
2. Write the result back into `v` using `v.copy_(v_new)` (atom: buffer-copy_-inplace).
3. Return `None`. The caller's reference to `v` MUST be mutated in place; `id(v)` MUST be unchanged.

Specifically: the test creates `v`, takes a snapshot of `id(v)`, calls `cx26_update_v_via_copy(v, g, 0.999)`, and asserts:
- `id(v)` is still the same (you did NOT rebind).
- `v.data_ptr()` is still the same storage (you did NOT detach / clone-and-overwrite).
- `v` element-wise equals `0.999 * v_before + 0.001 * g**2`.
- The function is robust to `g` having `requires_grad=True` (gradient should not flow through the buffer update — wrap with `t.no_grad()` if needed).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx26_update_v_via_copy(v, g, beta2):
    """In-place update v <- beta2*v + (1-beta2)*g**2 via copy_. Returns None."""
    raise NotImplementedError

def _test_cx26():
    # Case A: in-place semantics — id and data_ptr preserved.
    t.manual_seed(0)
    v = t.randn(3, 4)
    g = t.randn(3, 4)
    v_id_before = id(v)
    v_ptr_before = v.data_ptr()
    v_before = v.clone()
    beta2 = 0.999
    ret = cx26_update_v_via_copy(v, g, beta2)
    assert ret is None, 'function must return None (in-place semantics)'
    assert id(v) == v_id_before, 'v was rebound — must update in place'
    assert v.data_ptr() == v_ptr_before, 'v storage changed — copy_ should preserve data_ptr'
    expected = beta2 * v_before + (1 - beta2) * g * g
    assert t.allclose(v, expected, atol=1e-7), 'v did not get the EMA-of-squared-gradient update'

    # Case B: aliasing — buffer held in a dict sees the update too.
    v2 = t.zeros(5)
    stash = {'v_ref': v2}
    g2 = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    cx26_update_v_via_copy(v2, g2, beta2=0.5)
    # After: v = 0.5*0 + 0.5*g^2 = 0.5 * [1,4,9,16,25] = [0.5,2,4.5,8,12.5].
    assert t.allclose(stash['v_ref'], t.tensor([0.5, 2.0, 4.5, 8.0, 12.5]), atol=1e-7), (
        'dict-aliased reference did not see the update — proves you rebound instead of copy_'
    )

    # Case C: works when g has requires_grad=True — no autograd graph through v.
    v3 = t.ones(4)
    g3 = t.tensor([0.1, 0.2, 0.3, 0.4], requires_grad=True)
    cx26_update_v_via_copy(v3, g3, beta2=0.9)
    # v should be 0.9*1 + 0.1*g^2.
    expected3 = 0.9 * t.ones(4) + 0.1 * g3.detach() ** 2
    assert t.allclose(v3, expected3, atol=1e-7)
    # v itself must not require grad — a buffer is not a parameter.
    assert v3.requires_grad is False, 'v buffer should not require grad after the update'

    # Case D: multi-step EMA accumulates correctly.
    v4 = t.zeros(2)
    for step in range(3):
        g_step = t.tensor([1.0, 2.0])
        cx26_update_v_via_copy(v4, g_step, beta2=0.5)
    # After 3 steps with constant g and v0=0:
    # v1 = 0.5*0 + 0.5*g^2 = 0.5*g^2
    # v2 = 0.5*0.5*g^2 + 0.5*g^2 = 0.75*g^2
    # v3 = 0.5*0.75*g^2 + 0.5*g^2 = 0.875*g^2
    assert t.allclose(v4, 0.875 * t.tensor([1.0, 4.0]), atol=1e-7), (
        f'multi-step EMA broken; got {v4.tolist()}'
    )
    _dd_passed.add('cx26')

_test_cx26()

<details><summary>Show solution — cx26</summary>

```python
def cx26_update_v_via_copy(v, g, beta2):
    # No autograd through buffer updates.
    with t.no_grad():
        # Atom A (ema-second-moment): compute the new value out-of-place.
        v_new = beta2 * v + (1 - beta2) * g * g
        # Atom B (buffer-copy_-inplace): write back into the SAME storage.
        v.copy_(v_new)
```

The semantic difference between `v = v_new` and `v.copy_(v_new)` only matters when SOMEONE ELSE holds a reference to the original `v` — `register_buffer`, an optimizer's `state` dict, or just a separate variable. `copy_` writes into the existing storage; `=` rebinds the local name. PyTorch's `register_buffer` stores the tensor identity in the module's `_buffers` dict, so a rebind would orphan the registered version.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["Optimizer: Adam EMA second moment", "PyTorch: in-place buffer copy"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()